# IFRS 17 Term Life Engine - Demo Walkthrough

This notebook is a small, reviewer-friendly walkthrough of the portfolio-grade IFRS 17 Term Life valuation engine.

It uses a deliberately small configuration so the notebook can be run quickly. The project remains a simplified, educational IFRS 17 principles-aligned model, not a production IFRS 17 system or regulatory reporting tool.


## 1. Setup

Load project modules from the repository root and keep the demo output separate from normal engine outputs.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DEMO_OUTPUT = ROOT / "outputs" / "demo_notebook"
DEMO_OUTPUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
print(ROOT)


## 2. Config Load

Load the main config, then reduce policy count and disable heavy outputs for a fast walkthrough.


In [ ]:
from engine.assumptions import loadconfig

base_config = loadconfig(str(ROOT / "config" / "config.json"))
config = base_config.model_copy(
    update={
        "n_policies": 500,
        "projection_years": 10,
        "coverage_years": 10,
        "run_scenarios": False,
        "excel_output": False,
        "n_risk_scenarios": 100,
        "scenario_max_workers": 1,
        "output_path": str(DEMO_OUTPUT / "demo_projection.xlsx"),
    }
)

config.model_dump()


## 3. Master Data Preview

Create a deterministic synthetic portfolio using the configured random seed.


In [ ]:
from engine.projection import create_master_data

master = create_master_data(config)
master.head()


## 4. Projection Preview

Create policy-year projection rows with age, mortality, lapse, survival, and in-force fields.


In [ ]:
from engine.projection import create_projection_table, load_mortality_table
from engine.curves import create_discount_curve

mortality_table = load_mortality_table(str(ROOT / config.mortality_table_path)) if config.use_mortality_table else None
projection = create_projection_table(master, config, mortality_table)
discount_curve = create_discount_curve(config)
projection = projection.merge(discount_curve, on="Year", how="left", validate="m:1")

projection.head()


## 5. Cashflow Preview

Calculate premiums, claims, expenses, surrender benefits, reinsurance flows, and present value columns.


In [ ]:
from engine.cashflows import calculate_cashflows

projection = calculate_cashflows(projection, config)

cashflow_cols = [
    "policy_id",
    "Year",
    "Gross_Premium_Inflow",
    "Death_Benefits",
    "Operating_Expenses",
    "Reinsurance_Ceding",
    "Reinsurance_Recovery",
    "PV_Gross_Premium_Inflow",
    "PV_Death_Benefits",
]
projection[cashflow_cols].head()


## 6. BEL / RA / CSM Totals

Calculate liability-positive BEL, simplified RA, and CSM roll-forward.


In [ ]:
from engine.bel import calculate_bel, get_last_bel_diagnostic_summary
from engine.ra import calculate_risk_adjustment
from engine.csm import calculate_csm_rollforward
from engine.grouping import assign_ifrs17_groups

bel = calculate_bel(projection, config)
base_bel_diagnostics = get_last_bel_diagnostic_summary()
ra = calculate_risk_adjustment(projection, config)
csm = calculate_csm_rollforward(bel, ra, projection, config)
groups = assign_ifrs17_groups(csm, projection, config)

summary = pd.DataFrame(
    [
        {"metric": "Total BEL", "value": bel["bel_per_policy"].sum()},
        {"metric": "Total RA", "value": ra["ra_per_policy"].sum()},
        {"metric": "Total CSM Opening", "value": csm["csm_opening"].sum()},
        {"metric": "Total CSM Closing", "value": csm["csm_closing"].sum()},
        {"metric": "Onerous Loss", "value": csm["onerous_loss"].sum()},
        {"metric": "Groups", "value": len(groups)},
    ]
)
summary


## 7. Scenario Results

Run the standard scenarios on a smaller config and inspect total BEL / RA / CSM changes.


In [ ]:
from engine.scenarios import SCENARIOS, run_scenarios

scenario_config = config.model_copy(update={"n_policies": 200, "n_risk_scenarios": 50})
scenario_results = run_scenarios(scenario_config, SCENARIOS, mortality_table)
scenario_results


## 8. Audit JSON Example

Save demo outputs and inspect selected audit fields. The audit files include assumption snapshot, deterministic hash, row counts, reconciliation totals, and BEL diagnostics.


In [ ]:
from engine.outputs import save_outputs

save_outputs(
    projection=projection,
    bel_result=bel,
    ra_result=ra,
    csm_result=csm,
    scenario_results=scenario_results,
    output_dir=str(DEMO_OUTPUT),
    master=master,
    excel_output=False,
    group_result=groups,
    config=config,
    bel_diagnostics=base_bel_diagnostics,
)

with (DEMO_OUTPUT / "audit_report.json").open("r", encoding="utf-8") as f:
    audit_report = json.load(f)

{
    "run_id": audit_report["run_id"],
    "assumption_hash": audit_report["assumption_hash"],
    "row_counts": audit_report["row_counts"],
    "reconciliation_totals": audit_report["reconciliation_totals"],
}


## 9. Visual Checks

The charts below are simple reviewer aids, not formal model validation.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

bel["bel_per_policy"].plot(kind="hist", bins=30, ax=axes[0], title="BEL Distribution")
axes[0].set_xlabel("BEL per policy")

csm[["policy_id", "csm_opening", "csm_closing"]].set_index("policy_id").head(30).plot(
    kind="bar", ax=axes[1], title="CSM Opening vs Closing (first 30 policies)"
)
axes[1].set_ylabel("Amount")

scenario_results.set_index("scenario")["Total_BEL"].plot(
    kind="bar", ax=axes[2], title="Scenario Total BEL"
)
axes[2].set_ylabel("Total BEL")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


## 10. Reviewer Notes

Useful follow-up checks:

- Inspect `bel_diagnostics.json` for liability-positive BEL diagnostics.
- Compare scenario totals against the base run.
- Challenge discount timing, RA assumptions, CSM treatment, grouping logic, and data lineage.
- See `DOCUMENTS/methodology.md` and `DOCUMENTS/model_limitations.md` for scope boundaries.
